# G1 · Does the text-to-image transport work? (checklist steps 1.1–1.7)

Seam 1 moves every negative label from CLIP's text space into its image space with a map fitted on confident
ID images. G1 decides which map ReNeg uses — or whether we need Plan B — **before** Seam 2 is built on it.

| Part | Question | Uses labels? |
|---|---|---|
| A | Which test images are confident ID, and which ID classes have enough of them? | no (labels only reported) |
| B | Which map and λ predict *held-out* ID classes best? (leave-subtree-out) | no |
| C | Do mapped OOD class names land next to their real images? (NINCO, Textures, SSB-hard) | evaluation only |
| D | Does an image-space score beat the text-space score? | evaluation only |
| E | How many confident ID images does the map need? (cold start) | evaluation only |

**Runtime:** GPU recommended (*T4 or L4*; about 5–10 minutes). On a CPU it takes about 15–30 minutes.
**Output:** `MyDrive/ReNeg/oodlab_working/results/G1_transport_gate/` — `g1_report.md`, `g1_results.json`,
and one CSV per table.

In [ ]:
# ── Colab set-up (the same cell in every ReNeg notebook) ─────────────────────────────
import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
HOME = os.environ.get("RENEG_OODLAB_HOME", "/content/oodlab_home" if IN_COLAB else os.path.abspath("oodlab_home"))
for d in (f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/scratch", f"{HOME}/input"):
    os.makedirs(d, exist_ok=True)
if not os.path.islink(f"{HOME}/working"):              # working/ lives on Drive, scratch/ stays local
    if os.path.isdir(f"{HOME}/working") and not os.listdir(f"{HOME}/working"):
        os.rmdir(f"{HOME}/working")
    if not os.path.exists(f"{HOME}/working"):
        os.symlink(f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/working")
os.environ["OODLAB_HOME"] = HOME
# a newer ReNeg_Colab_bundle*.zip in uploads/ is installed here, so a new bundle needs no C0 re-run
import glob, re, shutil, zipfile
def _reneg_version(text):
    m = re.search(r'__version__ = "([0-9.]+)"', text or "")
    return tuple(int(x) for x in m.group(1).split(".")) if m else None
_init = f"{DRIVE_ROOT}/code/reneg_code/reneg/__init__.py"
_installed = _reneg_version(open(_init).read()) if os.path.isfile(_init) else None
_newest = None
for _zp in glob.glob(f"{DRIVE_ROOT}/uploads/*.zip"):
    try:
        with zipfile.ZipFile(_zp) as _z:
            _hit = [n for n in _z.namelist() if n.endswith("reneg_code/reneg/__init__.py")]
            _v = _reneg_version(_z.read(_hit[0]).decode()) if _hit else None
        if _v and (_newest is None or _v > _newest[0]):
            _newest = (_v, _zp, _hit[0][: -len("reneg_code/reneg/__init__.py")])
    except (zipfile.BadZipFile, OSError):
        pass
if _newest and (_installed is None or _newest[0] > _installed):
    _tmp = os.path.join(HOME, "scratch", "_bundle")
    shutil.rmtree(_tmp, ignore_errors=True)
    with zipfile.ZipFile(_newest[1]) as _z:
        _z.extractall(_tmp)
    _src = os.path.join(_tmp, _newest[2])
    shutil.rmtree(f"{DRIVE_ROOT}/code/reneg_code", ignore_errors=True)
    shutil.copytree(os.path.join(_src, "reneg_code"), f"{DRIVE_ROOT}/code/reneg_code")
    os.makedirs(f"{DRIVE_ROOT}/notebooks", exist_ok=True)
    for _f in os.listdir(os.path.join(_src, "notebooks")):
        if _f.endswith(".ipynb") and not _f.startswith("C0"):
            shutil.copy2(os.path.join(_src, "notebooks", _f), f"{DRIVE_ROOT}/notebooks/{_f}")
    _fmt = lambda v: ".".join(map(str, v)) if v else "none"
    print(f"reneg code updated {_fmt(_installed)} -> {_fmt(_newest[0])} from uploads/{os.path.basename(_newest[1])}")
    if "reneg" in sys.modules:
        print("reneg was already imported in this session: Runtime -> Restart session, then run again")
CODE_ROOT = f"{DRIVE_ROOT}/code/oodlab_code"           # same name as in the Kaggle notebooks
for p in (CODE_ROOT, f"{DRIVE_ROOT}/code/reneg_code"):
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} not found: run C0_transfer_from_kaggle first (GUIDE_COLAB.md, step 4)")
    if p not in sys.path:
        sys.path.insert(0, p)
import oodlab, reneg
print(f"oodlab {oodlab.__version__} | reneg {reneg.__version__} | project folder {DRIVE_ROOT}")

In [ ]:
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True)
ctx = start("G1_transport_gate", need_gpu=False)
log = ctx.log
import json, time
import numpy as np, pandas as pd, torch
from oodlab.report import find_cache
from reneg.g1 import G1, G1Config, clip_text_encoder
from reneg.wordnet import get_wordnet
pd.set_option("display.width", 200)

BACKBONE = "ViT-B/16"
cache = find_cache(ctx, BACKBONE)
bank = cache.load_textbank()
wn = get_wordnet()
print("WordNet:", "ready" if wn is not None else "unavailable -> text-cluster groups; SSB-hard names skipped")

### Settings

The defaults are the spec's starting values. Change them only for a reason you would write in the paper;
every value is saved with the results.

In [ ]:
cfg = G1Config(
    backbone=BACKBONE,
    device=ctx.device,
    id_stream_set="imagenet_test",   # confident ID images for fitting (45,000 test images, no labels used)
    p_min=0.5,                       # checklist 1.1 fallback: 0.3
    n_min=3, n_cap=20,
    eps=0.01, eps_align=0.01,        # a richer map must win by 1 point of top-1, or tie and gain 0.01 cosine
)
g = G1(cache, bank, cfg, logger=log)
print(json.dumps({k: v for k, v in cfg.__dict__.items() if k not in ("protocols",)}, indent=1, default=str))

## Part A · Confident ID images (checklist 1.1)

**Pass:** at least 900 of the 1,000 classes have 3 or more confident images. **If not:** set `p_min=0.3` above.

In [ ]:
a = g.part_a()
print(json.dumps(a, indent=1))
print("PASS" if a["pass"] else "FAIL -> lower p_min to 0.3 in the settings cell and re-run from there")

## Part B · Self-check on ID classes (checklist 1.2)

Whole WordNet subtrees (dogs, birds, vehicles, …) are held out in turn. Each candidate map is fitted on the
rest, exactly as the method would fit it (pseudo-labelled stream images, no labels), and must predict the
held-out classes' image centroids from their names. The held-out centroids come from OpenOOD's **labelled
validation split** (5,000 ImageNet images, the split meant for tuning): pseudo-labelled centroids would favour
raw text, because CLIP picked those images by their similarity to the class name. No OOD data is used.

**Pass:** some map is clearly better than raw text (R0). The line `online_choice` shows what the method would
pick on its own at test time (pseudo-labels, alignment first); ideally it is the same kind of map.

In [ ]:
b = g.part_b(wn)
print(json.dumps({k: v for k, v in b.items() if k != "best_per_kind"}, indent=1, default=str))
print("best of each kind:", b["best_per_kind"])
g.tables["selfcheck"].head(15).round(4)

## Part C · OOD concepts (checklist 1.3)

The names of NINCO, Textures and SSB-hard classes are mapped and compared with those classes' real image
centroids. These labels only evaluate; nothing is tuned on them. **Pass:** the chosen map beats R0 on 2 of 3
sets and its alignment never drops.

In [ ]:
from oodlab.clipwrap import load_clip
WEIGHTS = f"{DRIVE_ROOT}/weights"
model = load_clip(BACKBONE, device=ctx.device, input_roots=ctx.input_roots + [WEIGHTS], download_root=WEIGHTS)
enc = clip_text_encoder(model, prompt=cfg.prompt)
c = g.part_c(enc, wn)
print(json.dumps({k: v for k, v in c.items() if k != "previews"}, indent=1, default=str))
for p in c.get("previews", {}).values():
    print(p)
ct = g.tables["concepts"]
ct[ct.candidate.isin(["R0", "R1", g.chosen_name])].round(4)

## Part D · Image-space score vs text-space score (checklist 1.4)

NegLabel's score with the 10,000 negatives in text space, against the same score with transported
prototypes (`T`) or ID centroids plus transported negatives (`C`), alone (ω = 1) or fused. **Pass:** near-OOD
FPR95 drops by at least 1 point and far-OOD rises by at most 1 point.

In [ ]:
d = g.part_d()
print(json.dumps(d, indent=1, default=str))
s = g.tables["scores_summary"]
s[s.protocol == "openood_v15"].pivot_table(index=["candidate", "protos", "omega"], columns="group", values="fpr95").round(2).sort_values("near").head(15)

## Part E · Cold start (checklist 1.5)

The map is refitted from the first *n* confident images of a shuffled ID stream and scored on the tuning split
(ImageNet validation vs OpenImage-O validation). **Pass:** the fused score beats text before 2,048 images.

In [ ]:
e = g.part_e()
print(json.dumps(e, indent=1, default=str))
g.tables["coldstart"].round(3)

## Decision (checklist 1.7)

In [ ]:
dec = g.decide()
path = g.save(ctx.out)
print(json.dumps({k: v for k, v in dec.items() if k != "config"}, indent=1))
print("\nReport:", path)
finish(ctx, {"verdict": dec["verdict"], "chosen": dec["chosen"]})

In [ ]:
import matplotlib.pyplot as plt
sc = g.tables["selfcheck"]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for kind, sub in sc[sc.kind.isin(["R2", "B1", "B2"])].groupby("kind"):
    best = sub.sort_values("top1").groupby("lam").tail(1).sort_values("lam")
    ax[0].plot(best.lam, 100 * best.top1, marker="o", label=kind)
    ax[1].plot(best.lam, best["align"], marker="o", label=kind)
for k in ("R0", "R1"):
    r = sc[sc.candidate == k].iloc[0]
    ax[0].axhline(100 * r["top1"], ls="--", c="grey" if k == "R0" else "k", label=k)
    ax[1].axhline(r["align"], ls="--", c="grey" if k == "R0" else "k", label=k)
ax[0].set(xscale="log", xlabel="λ (relative)", ylabel="held-out top-1 (%)", title="Self-check: discrimination")
ax[1].set(xscale="log", xlabel="λ (relative)", ylabel="cosine to real centroid", title="Self-check: alignment")
cs = g.tables.get("coldstart")
if cs is not None and len(cs):
    cs = cs[cs.n_seen > 0]
    ax[2].plot(cs.n_seen, cs.fpr95_text, "--", c="grey", label="text")
    ax[2].plot(cs.n_seen, cs.fpr95_fused, marker="o", label="fused")
    ax[2].set(xscale="log", xlabel="confident ID images seen", ylabel="FPR95 (%)", title="Cold start")
for a_ in ax:
    a_.legend(fontsize=8)
plt.tight_layout()
plt.savefig(ctx.path("g1_plots.png"), dpi=120)
plt.show()

### What to do with the verdict

* **PASS** — mark checklist steps 1.1–1.7 Done, write the chosen map, λ and κ* into step 1.7, and start Seam 2.
* **PARTIAL** — the map generalises but the static score gain is small: keep it for screening near negatives
  (Seam 2) and re-test the score inside TANL at G2.
* **R1-ONLY** — use the gap shift, and try Plan B for a richer map (checklist 1.6).
* **FAIL** — run `G1b_transport_diagnostics` first (checklist 1.5b). It re-tests the maps in the direction the
  score uses, with tuned temperatures, and tells whether Plan B (a learned prior on ViT-L/14) could pay at all.

`g1_report.md` summarises the gate's decision.

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")